### Import needed libraries

In [50]:
import os
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import LSTM, Dense, GRU,  Masking, Dropout, BatchNormalization # type: ignore
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, # type: ignore
                                      TensorBoard, LearningRateScheduler, )

os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/cuda'


In [32]:
X = []
y = []

data_dir = "pickles/"

for file in os.listdir(data_dir):
    file_path = os.path.join(data_dir, file)
    if os.path.exists(file_path):
        with open(file_path, "rb") as f:
            data = pickle.load(f)
        for entry in data:
            points = entry["points"]  # list of frames, each frame = list of (x,y)
            if points:
                # Flatten each frame into 1D vector
                seq = [np.array(frame, dtype=np.float32).flatten() for frame in points]
                X.append(seq)
                y.append(entry["class_name"])
    else:
        print(f"{file} not found!")

print(f"Loaded {len(X)} sequences")

Loaded 124 sequences


In [57]:
max_seq_len = max(len(seq) for seq in X)
feature_dim = max(len(frame) for seq in X for frame in seq)  # largest frame vector size

X_padded = []
for seq in X:
    arr = np.zeros((max_seq_len, feature_dim), dtype=np.float32)
    for i, frame in enumerate(seq):
        arr[i, :len(frame)] = frame
    X_padded.append(arr)

X_padded = np.array(X_padded, dtype=np.float32)  # (num_samples, max_seq_len, feature_dim)
print("X_padded shape:", X_padded.shape)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_onehot = to_categorical(y_encoded)

print(X_padded.shape, y_onehot.shape)
print(X_padded[0])
print(X_padded[0][0])

X_padded shape: (124, 20, 468)
(124, 20, 468) (124, 6)
[[228.42125 241.90796 230.21948 ...   0.        0.        0.     ]
 [228.20882 241.82831 230.02475 ...   0.        0.        0.     ]
 [228.74829 241.68411 230.59627 ...   0.        0.        0.     ]
 ...
 [229.10213 240.91397 230.91934 ...   0.        0.        0.     ]
 [228.93362 240.84071 230.76549 ...   0.        0.        0.     ]
 [230.08621 240.0431  231.89656 ...   0.        0.        0.     ]]
[228.42125 241.90796 230.21948 243.70619 230.21948 243.70619 232.91682
 245.50443 232.91682 245.50443 237.4124  248.20177 237.4124  248.20177
 243.70619 250.      243.70619 250.      250.      250.      250.
 250.      256.2938  249.10089 256.2938  249.10089 261.68848 247.30266
 261.68848 247.30266 266.18408 245.50443 266.18408 245.50443 269.78052
 243.70619 269.78052 243.70619 271.57877 241.90796 228.42125 241.90796
 229.32036 240.10974 229.32036 240.10974 232.0177  237.4124  232.0177
 237.4124  236.51328 234.71504 236.51328 234.7

In [67]:
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y_onehot, test_size=0.3, random_state=42, stratify=y_encoded
)

from tensorflow.keras.layers import Bidirectional # type: ignore

model = Sequential([
    Masking(mask_value=0.0, input_shape=(X_padded.shape[1], X_padded.shape[2])),
    Bidirectional(LSTM(64, return_sequences=True)),
    BatchNormalization(),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(y_onehot.shape[1], activation="softmax")
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True) # type: ignore
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5) # type: ignore

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr]
)

loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc:.3f}")


/home/yassin/miniconda3/envs/tf/lib/python3.13/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_33"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_32 (Masking)            │ (None, 20, 468)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_47                │ (None, 20, 128)        │       272,896 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_35          │ (None, 20, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_48                │ (None, 128)            │        98,816 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_61 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_50 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_62 (Dense)                │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 389,510 (1.49 MB)

 Trainable params: 389,254 (1.48 MB)

 Non-trainable params: 256 (1.00 KB)

Epoch 1/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step - accuracy: 0.3023 - loss: 1.6495 - val_accuracy: 0.2895 - val_loss: 1.6575 - learning_rate: 0.0010
Epoch 2/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5116 - loss: 1.3328 - val_accuracy: 0.5000 - val_loss: 1.5639 - learning_rate: 0.0010
Epoch 3/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6395 - loss: 1.1018 - val_accuracy: 0.5526 - val_loss: 1.4871 - learning_rate: 0.0010
Epoch 4/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7326 - loss: 0.9344 - val_accuracy: 0.2105 - val_loss: 1.5551 - learning_rate: 0.0010
Epoch 5/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6977 - loss: 0.8715 - val_accuracy: 0.4211 - val_loss: 1.2914 - learning_rate: 0.0010
Epoch 6/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8023 - loss: 0.7043 - val_accuracy: 0.6053 - val_loss: 1.2676 - learning_rate: 0.0010
Epoch 7/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8140 - loss: 0.7166 - val_accuracy: 

In [35]:
model.save("gesture_model.h5", save_format="h5")

In [36]:
# import libraries
with open('label_encoder.pkl', 'wb') as f:
  pickle.dump(le, f)

In [37]:
import tensorflow as tf
print("TF:", tf.__version__)
print("Physical GPUs:", tf.config.list_physical_devices("GPU"))
try:
    from tensorflow.python.client import device_lib
    print(device_lib.list_local_devices())
except Exception:
    pass


TF: 2.20.0
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 13672885071901688980
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5645991936
locality {
  bus_id: 1
  links {
  }
}
incarnation: 15202183309537230043
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6"
xla_global_id: 416903419
]


I0000 00:00:1758305451.957903    4688 gpu_device.cc:2020] Created device /device:GPU:0 with 5384 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6
